In [13]:
!pip install google-genai
!pip install pinecone
!pip install pinecone-text pypdf
!pip install rapidfuzz
!pip install tqdm
!pip install ipywidgets


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [28]:
# ==============================================================================
# 💡 LESSONS LEARNED
# ==============================================================================
# Pinecone calculates hybrid search by natively adding the dense and sparse dot
# products together. To weight the results, we multiply our Query (Q) vectors
# by alpha and 1 - alpha respectively. This mathematically shrinks their Euclidean magnitude
# without changing their semantic Cosine angle.
#
# Weighted Score = [alpha * ||Q_dense|| * ||D_dense|| * cos(θ_dense)] + [(1 - alpha) * ||Q_sparse|| * ||D_sparse|| * cos(θ_sparse)]
# ==============================================================================

from google import genai
from google.genai import types
from pinecone import Pinecone
from pinecone_text.sparse import BM25Encoder
from pypdf import PdfReader
from pydantic import BaseModel, Field
from tqdm import tqdm
from datetime import date
from enum import Enum
from rapidfuzz import process, fuzz
from dotenv import load_dotenv
import numpy as np
import csv
import io
import time
import random
import uuid
import logging
import os

load_dotenv()
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")

In [16]:
def load_taxonomy_skills(filepath: str) -> list[str]:
    skills = []
    try:
        with open(filepath, mode='r', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f)
            for row in reader:
                label = row.get('preferredLabel')
                if label:
                    skills.append(label)
    except Exception as e:
        print(f"Error loading taxonomy: {e}")
    return skills

def normalize_skills(raw_skills: list[str], taxonomy_skills: list[str], threshold: float = 80.0) -> list[str]:
    normalized = []
    for skill in raw_skills:
        best_match = process.extractOne(skill, taxonomy_skills, scorer=fuzz.WRatio)
        if best_match:
            match_string, match_score, _ = best_match
            if match_score >= threshold:
                normalized.append(match_string)
            else:
                normalized.append(skill)
    return list(set(normalized))

In [21]:
gemini_client = genai.Client()
pinecone_client = Pinecone()
pinecone_index = pinecone_client.Index('cjm-hybrid')
index_namespace = "matching-pool"

logging.getLogger("pypdf").setLevel(logging.ERROR)

class DocType(str, Enum):
    CV = "cv"
    VACANCY = "vacancy"

class CandidateMetadata(BaseModel):
    country: str = Field(description="Two character country code where the candidate resides, e.g. 'DE', 'CH', 'AT', 'US'. If unclear, set to 'UNKNOWN'")
    city: str = Field(description="The candidate's city of residence, e.g. 'Berlin', 'Bern', 'Vienna', 'New York'. If unclear, set to 'UNKNOWN'")
    career_level: str = Field(description="The candidates career level. Must be exactly one of these values: 'Junior', 'Mid', 'Senior', 'Lead'. If unclear, set to 'UNKNOWN'")
    department: str = Field(description="The candidates broad job department, e.g. 'Software Development', 'Data Science', 'Solution Architect', 'Tester'. If unclear, set to 'UNKNOWN'")
    raw_skills: list[str] = Field(description="A list of the core competencies of the candidate, e.g. tech stacks, tools, or methods. If unclear, return ['UNKNOWN']")
    highest_degree: str = Field(description="The highest level of education the candidate completed, e.g. 'Bachelor', 'Master', 'PhD', 'Apprenticeship'. If unclear, set to 'UNKNOWN'")
    salary_amount: int = Field(description="Expected annual gross salary of the candidate. Must be a plain integer. If unclear, set to 0")
    salary_currency: str = Field(description="ISO currency code of expected annual gross salary of the candidate, e.g. 'CHF', 'EUR', 'USD'. If unclear, 'UNKNOWN'")

class VacancyMetadata(BaseModel):
    country: str = Field(description="Two character country code where the vacancy is located, e.g. 'DE', 'CH', 'AT', 'US'. If unclear, set to 'UNKNOWN'")
    city: str = Field(description="The city where the vacancy is located, e.g. 'Berlin', 'Bern', 'Vienna', 'New York'. If unclear, set to 'UNKNOWN'")
    career_level: str = Field(description="Target career level for the vacancy. Must be exactly one of these values: 'Junior', 'Mid', 'Senior', 'Lead'. If unclear, set to 'UNKNOWN'")
    department: str = Field(description="Broad job department the the vacancy can be assigned to, e.g. 'Software Development', 'Data Science', 'Solution Architect', 'Tester'. If unclear, set to 'UNKNOWN'")
    raw_skills: list[str] = Field(description="A list of the critical core competencies required for the vacancy, e.g. tech stacks, tools, or methods. If unclear, return ['UNKNOWN']")
    highest_degree: str = Field(description="Minimum education required for the vacancy, e.g. 'Bachelor', 'Master', 'PhD', 'Apprenticeship'. If unclear, set to 'UNKNOWN'")
    salary_amount: int = Field(description="Maximum annual gross salary for the vacancy. Must be a plain integer. If unclear, set to 0")
    salary_currency: str = Field(description="ISO currency code of maximum annual gross salary for the vacancy, e.g. 'CHF', 'EUR', 'USD'. If unclear, 'UNKNOWN'")

cv_prompt = """
    You are an HR data extraction system.

    Extract structured information from the CV according to the provided schema.

    Rules:
    - Use ONLY information explicitly stated in the CV
    - Do NOT infer or guess missing data

    Return a valid JSON matching the schema. No explanations required.
    """

vacancy_prompt = """
    You are an HR data extraction system.

    Extract structured information from the vacancy according to the provided schema.

    Rules:
    - Use ONLY information explicitly stated in the vacancy
    - Do NOT infer or guess missing data

    Return a valid JSON matching the schema. No explanations required.
    """

def extract_text_from_pdf(reader: PdfReader) -> str:
    text = ""
    for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + " "
    return text.strip()

def fit_bm25(documents: list[object], output_file="bm25_weights.json"):
    pbar = tqdm(documents, desc=f"Augmenting documents to fit bm25", unit="doc")

    corpus = []
    for document in pbar:
        reader = PdfReader(io.BytesIO(document["pdf_bytes"]))
        raw_text = extract_text_from_pdf(reader)
        augmented = f"{raw_text} [TAGS: {' '.join(document['normalized_skills'])}]"
        corpus.append(augmented)

    pbar.close()
    bm25 = BM25Encoder().default()
    bm25.fit(corpus)
    bm25.dump(output_file)

def infer_metadata(folder_path: str, prompt: str, metadata_schema: type[BaseModel], taxonomy_skills: list[str]):
    files = [fn for fn in sorted(os.listdir(folder_path)) if fn.lower().endswith('.pdf')]
    pbar = tqdm(files, desc=f"Inferring metadata of documents", unit="doc")

    enriched_documents = []
    for fn in pbar:
        with open(os.path.join(folder_path, fn), "rb") as f:
            pdf_bytes = f.read()
        pdf_part = types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf')

        inferred_metadata = retry_call(lambda: gemini_client.models.generate_content(
        model='gemini-3.5-flash',
        contents=[pdf_part, prompt],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=metadata_schema,
            temperature=0.0
            ),
        ))

        validated_metadata = metadata_schema.model_validate_json(inferred_metadata.text).model_dump()
        validated_metadata["normalized_skills"] = normalize_skills(validated_metadata.get("raw_skills", []), taxonomy_skills)
        validated_metadata["filename"] = fn
        validated_metadata["pdf_bytes"] = pdf_bytes
        enriched_documents.append(validated_metadata)

    pbar.close()
    return enriched_documents

def embed(documents: list[object], doc_type: DocType):
    pbar = tqdm(documents, desc=f"Embedding documents", unit="doc")

    bm25 = BM25Encoder().load("bm25_weights.json")
    records_to_upsert = []
    for document in pbar:
        pdf_part = types.Part.from_bytes(data=document["pdf_bytes"], mime_type='application/pdf')
        inferred_dense_embedding = retry_call(lambda: gemini_client.models.embed_content(
            model="gemini-embedding-2",
            contents=[pdf_part],
            config=types.EmbedContentConfig(
                output_dimensionality=768,
                task_type = "RETRIEVAL_DOCUMENT" if doc_type == DocType.CV else "RETRIEVAL_QUERY"
            )
        ))
        raw_text = extract_text_from_pdf(PdfReader(io.BytesIO(document["pdf_bytes"])))
        augmented_text = f"{raw_text} [TAGS: {' '.join(document['normalized_skills'])}]"
        inferred_sparse_embedding = bm25.encode_documents(augmented_text)

        records_to_upsert.append({
            "id": str(uuid.uuid4()),
            "values": inferred_dense_embedding.embeddings[0].values,
            "sparse_values": inferred_sparse_embedding,
            "metadata": {
                "type": doc_type,
                "filename": document["filename"],
                "last_updated": date.today().isoformat(),
                **{k: v for k, v in document.items() if k not in ["pdf_bytes", "filename"]},
            }
        })
    pbar.close()

    for i in range(0, len(records_to_upsert), 25):
        pinecone_index.upsert(vectors=records_to_upsert[i:i + 25], namespace=index_namespace)

def retry_call(fn, max_retries=3, base_delay=1.0):
    for attempt in range(max_retries):
        try:
            return fn()

        except Exception as e:
            wait = base_delay * (2 ** attempt) + random.uniform(0, 0.5)
            print(f"[RETRY] attempt {attempt+1}/{max_retries} failed: {e}. retrying in {wait:.2f}s")
            time.sleep(wait)

    raise RuntimeError(f"max retries of {max_retries} exceeded")

In [26]:
taxonomy_skills = load_taxonomy_skills("taxonomy/skills_en.csv")
cvs = infer_metadata(folder_path="cv", metadata_schema=CandidateMetadata,prompt=cv_prompt,taxonomy_skills=taxonomy_skills)
vacancies = infer_metadata(folder_path="vacancy", metadata_schema=CandidateMetadata, prompt=cv_prompt, taxonomy_skills=taxonomy_skills)
fit_bm25(cvs + vacancies)
embed(cvs, DocType.CV)
embed(vacancies, DocType.VACANCY)

Inferring metadata of documents:  88%|████████▊ | 7/8 [02:35<00:20, 20.73s/doc]

[RETRY] attempt 1/3 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}. retrying in 1.11s


Embedding documents: 100%|██████████| 1/1 [00:01<00:00,  1.69s/doc]


In [27]:
vacancies = pinecone_index.query(
    namespace=index_namespace,
    vector=[0.0] * 768,
    top_k=1,
    include_metadata=True,
    include_values=True,
    filter={"type": {"$eq": "vacancy"}}
)

vacancy = vacancies["matches"][0]
vacancy_dense_embedding = vacancy["values"]
vacancy_sparse_embedding = vacancy["sparse_values"]
vacancy_metadata = vacancy["metadata"]

alpha = 0.7

def hybrid_score(dense, sparse, alpha: float):
    if alpha < 0 or alpha > 1:
        raise ValueError("Alpha must be between 0 and 1")

    np_dense = np.array(dense)
    np_sparse = np.array(sparse["values"])

    scaled_dense = np_dense * alpha
    scaled_sparse = np_sparse * (1 - alpha)

    w_dense = scaled_dense.tolist()
    w_sparse = {
        "indices": sparse["indices"],
        "values": scaled_sparse.tolist()
    }
    return w_dense, w_sparse

scaled_dense, scaled_sparse = hybrid_score(vacancy_dense_embedding, vacancy_sparse_embedding, alpha)

matched_candidates = pinecone_index.query(
    namespace=index_namespace,
    vector=scaled_dense,
    sparse_vector=scaled_sparse,
    top_k=8,
    include_metadata=True,
    filter={
        "$and": [
            {"type": {"$eq": "cv"}}
        ]
    }
)

for candidate in matched_candidates["matches"]:
        score = candidate["score"]
        metadata = candidate["metadata"]
        print(f"Candidate: {metadata.get('filename')}")
        print(f"Score: {score}")

Candidate: senior_system_administrator.pdf
Score: 5.49691963
Candidate: junior_business_analyst.pdf
Score: 5.22716522
Candidate: senior_infrastructure_architect.pdf
Score: 5.04787493
Candidate: director_it.pdf
Score: 5.0077858
Candidate: senior_consultant.pdf
Score: 4.49173737
Candidate: senior_data_scientist.pdf
Score: 4.31694889
Candidate: junior_developer.pdf
Score: 4.03453827
Candidate: senior_culinary_chef.pdf
Score: 3.99367499
